# Databricks CLI setup (Windows)

Run the PowerShell commands below in a **PowerShell terminal**. Keep PATs out of notebooks, source control, and command history.

## 1. Create and activate the Python environment

Creation is shown for a new machine. Skip it when `%USERPROFILE%\dataeng` already exists. Activation changes the terminal session; it cannot persist when run as a notebook subprocess.

```powershell
py -m venv "$env:USERPROFILE\dataeng"
Set-ExecutionPolicy -Scope Process -ExecutionPolicy Bypass
& "$env:USERPROFILE\dataeng\Scripts\Activate.ps1"
python -m pip install --upgrade pip ipykernel
python -m ipykernel install --user --name dataeng --display-name "Python (dataeng)"
```

Select **Python (dataeng)** as this notebook's kernel.

In [ ]:
import os, sys
print("Python:", sys.executable)
print("Expected environment:", os.path.join(os.path.expanduser("~"), "dataeng"))

## 2. Install the current Databricks CLI

The current CLI (0.205+) is a standalone program used by Databricks Asset Bundles; it is not the legacy `pip install databricks-cli` package. Run in PowerShell, then restart the terminal and notebook application so PATH is refreshed.

```powershell
winget search Databricks
winget install --id Databricks.DatabricksCLI --exact
databricks version
```

In [ ]:
import shutil, subprocess
cli = shutil.which("databricks")
assert cli, "Databricks CLI not found. Restart Jupyter after installing it."
subprocess.run([cli, "version"], check=True)

## 3. Configure PAT authentication

Create a fresh PAT in the workspace UI. The command prompts for it securely; paste the PAT only at that prompt. A username is not required for PAT authentication.

```powershell
databricks configure --host https://dbc-d2625d47-674e.cloud.databricks.com --profile course-free
```

Credentials are stored in `%USERPROFILE%\.databrickscfg`. Do not add that file to a project or commit it.

## 4. Verify the profile and workspace access

```powershell
databricks auth profiles
databricks current-user me --profile course-free
databricks workspace list / --profile course-free
```

In [ ]:
profile = "course-free"
for args in (["current-user", "me"], ["workspace", "list", "/"]):
    result = subprocess.run([cli, *args, "--profile", profile], text=True, capture_output=True)
    print(f"$ databricks {' '.join(args)} --profile {profile}")
    print(result.stdout if result.returncode == 0 else result.stderr)
    result.check_returncode()

## 5. PAT maintenance

If a PAT is pasted into chat, a notebook, or source control, revoke it in Databricks, create a replacement, and rerun the configure command.